In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point

In [6]:
import h3

In [7]:
lights = gpd.read_parquet(
    "../data/processed/clean_streetlights.parquet"
)

crime = gpd.read_parquet(
    "../data/processed/clean_crime.parquet"
)

In [8]:
print("Streetlights:", lights.shape)
print("Crime:", crime.shape)

print("\nStreetlight CRS:", lights.crs)
print("Crime CRS:", crime.crs)

Streetlights: (108030, 46)
Crime: (63194, 39)

Streetlight CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon"

In [9]:
print("Streetlight columns:")
print(lights.columns.tolist())

Streetlight columns:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'facility_type', 'status', 'due_date', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'vehicle_type', 'taxi_company_borough', 'taxi_pick_up_location', 'bridge_highway_name', 'bridge_highway_direction', 'road_ramp', 'bridge_highway_segment', 'latitude', 'longitude', 'location', 'outage_duration_hours', 'geometry']


In [10]:
print("\nCrime columns:")
print(crime.columns.tolist())


Crime columns:
['cmplnt_num', 'cmplnt_fr_dt', 'cmplnt_fr_tm', 'cmplnt_to_dt', 'cmplnt_to_tm', 'addr_pct_cd', 'rpt_dt', 'ky_cd', 'ofns_desc', 'pd_cd', 'pd_desc', 'crm_atpt_cptd_cd', 'law_cat_cd', 'boro_nm', 'loc_of_occur_desc', 'prem_typ_desc', 'juris_desc', 'jurisdiction_code', 'parks_nm', 'hadevelopt', 'housing_psa', 'x_coord_cd', 'y_coord_cd', 'susp_age_group', 'susp_race', 'susp_sex', 'transit_district', 'latitude', 'longitude', 'lat_lon', 'patrol_boro', 'station_name', 'vic_age_group', 'vic_race', 'vic_sex', 'crime_datetime', 'crime_time', 'is_nighttime', 'geometry']


In [11]:
lights_2263 = lights.to_crs(epsg=2263)
crime_2263 = crime.to_crs(epsg=2263)

In [12]:
print(lights_2263.crs)
print(crime_2263.crs)

EPSG:2263
EPSG:2263


In [15]:
lights_wgs84 = lights.to_crs(epsg=4326)
crime_wgs84 = crime.to_crs(epsg=4326)

In [16]:
def get_h3_indices(gdf):
    gdf = gdf.copy()

    gdf["h3_res9"] = gdf.geometry.apply(
        lambda geom: h3.latlng_to_cell(
            geom.y,
            geom.x,
            9
        )
    )

    gdf["h3_res10"] = gdf.geometry.apply(
        lambda geom: h3.latlng_to_cell(
            geom.y,
            geom.x,
            10
        )
    )

    return gdf

In [17]:
lights_wgs84 = get_h3_indices(lights_wgs84)
crime_wgs84 = get_h3_indices(crime_wgs84)

In [21]:
lights_wgs84[["h3_res9", "h3_res10"]].head()


,h3_res9,h3_res10
0,892a1072d6fffff,8a2a1072d6dffff
1,892a100dd5bffff,8a2a100dd5affff
2,892a100dd13ffff,8a2a100dd127fff
3,892a10729b3ffff,8a2a10729b37fff
4,892a10729d7ffff,8a2a10729d67fff


In [22]:
crime_wgs84[["h3_res9", "h3_res10"]].head()

,h3_res9,h3_res10
0,892a1076673ffff,8a2a10766707fff
1,892a10772b7ffff,8a2a10772b6ffff
2,892a100d817ffff,8a2a100d814ffff
3,892a1038557ffff,8a2a1038555ffff
4,892a1074143ffff,8a2a1074140ffff


In [23]:
lights_2263["h3_res9"] = lights_wgs84["h3_res9"].values
lights_2263["h3_res10"] = lights_wgs84["h3_res10"].values

crime_2263["h3_res9"] = crime_wgs84["h3_res9"].values
crime_2263["h3_res10"] = crime_wgs84["h3_res10"].values

In [24]:
print(lights_2263[["h3_res9", "h3_res10"]].head())

           h3_res9         h3_res10
0  892a1072d6fffff  8a2a1072d6dffff
1  892a100dd5bffff  8a2a100dd5affff
2  892a100dd13ffff  8a2a100dd127fff
3  892a10729b3ffff  8a2a10729b37fff
4  892a10729d7ffff  8a2a10729d67fff


In [25]:
lights_2263["buffer_100m"] = lights_2263.geometry.buffer(100)

In [26]:
lights_2263["buffer_250m"] = lights_2263.geometry.buffer(250)

In [27]:
lights_2263["buffer_100m_inner"] = lights_2263.geometry.buffer(100)

lights_2263["buffer_250m_outer"] = lights_2263.geometry.buffer(250)

lights_2263["displacement_ring"] = (
    lights_2263["buffer_250m_outer"]
    .difference(lights_2263["buffer_100m_inner"])
)

In [28]:
treatment_zones = lights_2263[
    ["created_date", "closed_date", "geometry", "h3_res9", "h3_res10"]
].copy()

treatment_zones["geometry"] = treatment_zones.geometry.buffer(100)

In [29]:
crime_100m = gpd.sjoin(
    crime_2263,
    treatment_zones,
    how="inner",
    predicate="within"
)

In [30]:
print(crime_100m.shape)

(47694, 46)


In [31]:
displacement_zones = lights_2263[
    ["created_date", "closed_date", "geometry", "h3_res9", "h3_res10"]
].copy()

displacement_zones["geometry"] = (
    displacement_zones.geometry.buffer(250)
    .difference(displacement_zones.geometry.buffer(100))
)

In [32]:
crime_250m = gpd.sjoin(
    crime_2263,
    displacement_zones,
    how="inner",
    predicate="within"
)

In [33]:
print(crime_250m.shape)

(208635, 46)


In [34]:
lights_2263["created_date"] = pd.to_datetime(
    lights_2263["created_date"]
)

lights_2263["closed_date"] = pd.to_datetime(
    lights_2263["closed_date"]
)

crime_2263["crime_datetime"] = pd.to_datetime(
    crime_2263["crime_datetime"]
)

In [35]:
crime_100m["crime_datetime"] = pd.to_datetime(
    crime_100m["crime_datetime"]
)

crime_100m = crime_100m[
    (crime_100m["crime_datetime"] >= crime_100m["created_date"])
    &
    (crime_100m["crime_datetime"] <= crime_100m["closed_date"])
].copy()

In [36]:
crime_250m["crime_datetime"] = pd.to_datetime(
    crime_250m["crime_datetime"]
)

crime_250m = crime_250m[
    (crime_250m["crime_datetime"] >= crime_250m["created_date"])
    &
    (crime_250m["crime_datetime"] <= crime_250m["closed_date"])
].copy()

In [37]:
crime_counts_100m = (
    crime_100m
    .groupby("index_right")
    .size()
    .rename("crimes_during_outage_100m")
)

In [38]:
crime_counts_250m = (
    crime_250m
    .groupby("index_right")
    .size()
    .rename("crimes_during_outage_250m")
)

In [39]:
linked = lights_2263.copy()

In [40]:
linked["outage_index"] = linked.index

In [41]:
linked = linked.merge(
    crime_counts_100m,
    left_on="outage_index",
    right_index=True,
    how="left"
)

In [42]:
linked = linked.merge(
    crime_counts_250m,
    left_on="outage_index",
    right_index=True,
    how="left"
)

In [43]:
linked["crimes_during_outage_100m"] = (
    linked["crimes_during_outage_100m"]
    .fillna(0)
    .astype(int)
)

linked["crimes_during_outage_250m"] = (
    linked["crimes_during_outage_250m"]
    .fillna(0)
    .astype(int)
)

In [44]:
print(
    linked[
        [
            "created_date",
            "closed_date",
            "crimes_during_outage_100m",
            "crimes_during_outage_250m"
        ]
    ].head(20)
)

          created_date         closed_date  crimes_during_outage_100m  \
0  2026-09-16 21:39:00 2026-09-17 09:34:00                          0   
1  2026-09-15 20:07:00 2026-09-17 09:17:00                          0   
2  2026-09-15 15:54:00 2026-09-17 09:18:00                          0   
3  2026-09-15 12:47:00 2026-09-18 09:15:00                          0   
4  2026-09-15 09:35:00 2026-09-17 09:32:00                          0   
5  2026-09-14 22:45:00 2026-09-17 22:00:00                          0   
6  2026-09-14 15:34:00 2026-09-17 09:35:00                          0   
7  2026-09-14 15:34:00 2026-09-17 09:35:00                          0   
8  2026-09-14 05:38:00 2026-09-17 05:00:00                          0   
9  2026-09-13 21:19:00 2026-09-17 09:31:00                          0   
10 2026-09-13 20:29:00 2026-09-17 09:33:00                          0   
11 2026-09-12 23:03:00 2026-09-15 10:17:00                          0   
12 2026-09-11 12:21:00 2026-09-15 12:00:00         

In [45]:
print(
    linked[
        [
            "crimes_during_outage_100m",
            "crimes_during_outage_250m"
        ]
    ].describe()
)

       crimes_during_outage_100m  crimes_during_outage_250m
count              108030.000000              108030.000000
mean                    0.005276                   0.028085
std                     0.106661                   0.454614
min                     0.000000                   0.000000
25%                     0.000000                   0.000000
50%                     0.000000                   0.000000
75%                     0.000000                   0.000000
max                    12.000000                  67.000000


In [46]:
print(
    "Outages with at least one crime within 100m:",
    (linked["crimes_during_outage_100m"] > 0).sum()
)

print(
    "Outages with at least one crime within 250m:",
    (linked["crimes_during_outage_250m"] > 0).sum()
)

Outages with at least one crime within 100m: 421
Outages with at least one crime within 250m: 1628


In [47]:
linked = linked.to_crs(epsg=4326)

In [48]:
columns_to_drop = [
    "buffer_100m",
    "buffer_100m_inner",
    "buffer_250m_outer",
    "displacement_ring",
    "outage_index"
]

linked = linked.drop(
    columns=[c for c in columns_to_drop if c in linked.columns]
)

In [49]:
linked.to_parquet(
    "../data/processed/outage_crime_linked.parquet",
    index=False
)

In [50]:
test_linked = gpd.read_parquet(
    "../data/processed/outage_crime_linked.parquet"
)

print(test_linked.shape)

print(
    test_linked[
        [
            "crimes_during_outage_100m",
            "crimes_during_outage_250m"
        ]
    ].head()
)

print(test_linked.crs)

(108030, 51)
   crimes_during_outage_100m  crimes_during_outage_250m
0                          0                          0
1                          0                          0
2                          0                          0
3                          0                          0
4                          0                          0
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening

In [51]:
print(linked.columns.tolist())

['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'facility_type', 'status', 'due_date', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'vehicle_type', 'taxi_company_borough', 'taxi_pick_up_location', 'bridge_highway_name', 'bridge_highway_direction', 'road_ramp', 'bridge_highway_segment', 'latitude', 'longitude', 'location', 'outage_duration_hours', 'geometry', 'h3_res9', 'h3_res10', 'buffer_250m', 'crimes_during_outage_100m', 'crimes_during_outage_250m']


In [52]:
print("100m crimes:")
print(test_linked["crimes_during_outage_100m"].describe())

print("\n250m crimes:")
print(test_linked["crimes_during_outage_250m"].describe())

100m crimes:
count    108030.000000
mean          0.005276
std           0.106661
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          12.000000
Name: crimes_during_outage_100m, dtype: float64

250m crimes:
count    108030.000000
mean          0.028085
std           0.454614
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          67.000000
Name: crimes_during_outage_250m, dtype: float64


In [53]:
print(
    "Rows with at least 1 crime within 100m:",
    (test_linked["crimes_during_outage_100m"] > 0).sum()
)

print(
    "Rows with at least 1 crime within 250m:",
    (test_linked["crimes_during_outage_250m"] > 0).sum()
)

Rows with at least 1 crime within 100m: 421
Rows with at least 1 crime within 250m: 1628


In [54]:
print(
    test_linked[
        (test_linked["crimes_during_outage_100m"] > 0) |
        (test_linked["crimes_during_outage_250m"] > 0)
    ][
        [
            "created_date",
            "closed_date",
            "crimes_during_outage_100m",
            "crimes_during_outage_250m"
        ]
    ].head(10)
)

             created_date         closed_date  crimes_during_outage_100m  \
15771 2025-12-29 17:09:00 2026-01-07 07:00:00                          0   
15788 2025-12-29 09:37:00 2026-01-08 09:00:00                          0   
15852 2025-12-27 08:16:00 2026-01-06 09:22:00                          0   
15858 2025-12-26 23:10:00 2026-01-05 11:03:00                          0   
15900 2025-12-26 12:12:00 2026-01-05 11:05:00                          0   
16033 2025-12-23 14:54:00 2025-12-29 01:30:00                          0   
16046 2025-12-23 13:07:00 2025-12-29 01:20:00                          0   
16048 2025-12-23 13:01:00 2025-12-30 07:00:00                          0   
16060 2025-12-23 10:31:00 2025-12-29 08:20:00                          0   
16125 2025-12-22 11:45:00 2026-02-03 11:47:00                          0   

       crimes_during_outage_250m  
15771                          1  
15788                          1  
15852                          1  
15858                  